In [2]:
# Simple self attention
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [3]:
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [4]:
input_key1 = inputs[0]
input_key1

tensor([0.4300, 0.1500, 0.8900])

In [5]:
result = torch.dot(input_query, input_key1)
result

tensor(0.9544)

In [6]:
query_vector = inputs[1]
attention_scores = torch.empty(inputs.shape[0])
for index, key_vector in enumerate(inputs):
    attention_scores[index] = torch.dot(key_vector, query_vector)
    
attention_scores

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [7]:
# own softmax
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x))
    print(e)
    return e / e.sum()

x = np.array([2.0, 1.0, 0.1])
e = np.exp(x)
e.sum()
#print(softmax(x))

np.float64(11.212508845465344)

In [8]:
attention_scores
attention_weights = torch.softmax(attention_scores, dim=0) # Normalising attention score so that sum = 1 so help in training optimisation
attention_weights

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [9]:
# attention_scores = query_vector dot key_vector 
# attention_weights = softmax(attention_scores)
# context_vector = sum(attension_weights * value_vector)

print(inputs.shape[0])
print(attention_weights)
context_vector = torch.zeros(inputs.shape[1])
for i, value_vector in enumerate(inputs):
    att_weight = attention_weights[i]
    print(att_weight)
    print(value_vector)
    context_vector = context_vector + att_weight * value_vector
    print(context_vector)

context_vector





6
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(0.1385)
tensor([0.4300, 0.1500, 0.8900])
tensor([0.0596, 0.0208, 0.1233])
tensor(0.2379)
tensor([0.5500, 0.8700, 0.6600])
tensor([0.1904, 0.2277, 0.2803])
tensor(0.2333)
tensor([0.5700, 0.8500, 0.6400])
tensor([0.3234, 0.4260, 0.4296])
tensor(0.1240)
tensor([0.2200, 0.5800, 0.3300])
tensor([0.3507, 0.4979, 0.4705])
tensor(0.1082)
tensor([0.7700, 0.2500, 0.1000])
tensor([0.4340, 0.5250, 0.4813])
tensor(0.1581)
tensor([0.0500, 0.8000, 0.5500])
tensor([0.4419, 0.6515, 0.5683])


tensor([0.4419, 0.6515, 0.5683])

In [10]:
# Simple self attention mechanism without trainable parameters

attension_scores = torch.zeros(size=(6,6))
attention_weights = torch.zeros(size=(6,6))
for i, query_vector in enumerate(inputs):
    for j, key_vector in enumerate(inputs):
        attension_scores[i][j] = torch.dot(query_vector, key_vector)
    attention_weights[i] = torch.softmax(attension_scores[i], dim=0)

print(f"attention weights using for loop: {attention_weights}")




attention weights using for loop: tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [11]:
## Same thing using matrix multiplication - IMPORTANT ==============================

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
attension_scores_m = inputs @ inputs.T
attention_weights_m = torch.softmax(attension_scores_m, dim=1)
#print(f"attention weights matrix multiplication: {attention_weights_m}")
#print(f"inputs : {inputs}")

context_vector =  attention_weights_m @ inputs
print(f"context_vector : {context_vector}")

context_vector : tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [12]:
## Implement self attention with trainable weights - IMPORTANT ==============================
import torch.nn as nn

d_in = inputs.shape[1]
d_out = 2


# Initialize weight matrix for q,k,v
torch.manual_seed(123)
W_query = nn.Parameter(torch.rand(d_in, d_out))
W_key = nn.Parameter(torch.rand(d_in, d_out))
W_value = nn.Parameter(torch.rand(d_in, d_out))



# Calculate q,k,v mareix 
query_matrix = inputs @ W_query
key_matrix = inputs @ W_key
value_matrix = inputs @ W_value 

# scaling factor 
d_k = key_matrix.shape[1]


# Calculate attention weights
attension_scores = query_matrix @ key_matrix.T
attention_weights = torch.softmax(attension_scores/d_k**0.5, dim=1)
attention_weights

# Calculate Context matrix
context_matrix = attention_weights @ value_matrix
context_matrix

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [13]:
# Generalizing everything uptill now

d_in = inputs.shape[1]
d_out = 2

class MySelfAttention_V1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        torch.manual_seed(123)
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, inputs):
        # Calculate q,k,v mareix 
        query_matrix = inputs @ self.W_query
        key_matrix = inputs @ self.W_key
        value_matrix = inputs @ self.W_value 

        # scaling factor 
        d_k = key_matrix.shape[1]


        # Calculate attention weights
        attension_scores = query_matrix @ key_matrix.T
        attention_weights = torch.softmax(attension_scores/d_k**0.5, dim=1)
        attention_weights

        # Calculate Context matrix
        context_matrix = attention_weights @ value_matrix
        return context_matrix
    
self_attention = MySelfAttention_V1(d_in, d_out)
self_attention(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [14]:
# Generalizing everything with BETTER approach - with LINEAR LAYERS and CAUSAL MASKING and DROPOUT MASK

torch.manual_seed(123)

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
batch = torch.stack((inputs, inputs), dim=0)

print(batch)

d_in = inputs.shape[1]
d_out = 2

class MyCausalSelfAttention_V2(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout = 0.5, bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=bias)
        self.W_key = nn.Linear(d_in, d_out, bias=bias)
        self.W_value = nn.Linear(d_in, d_out, bias=bias)
        self.dropout_mask = nn.Dropout(p=dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, inputs):
        b, num_token, d_in = inputs.shape
        # Calculate q,k,v mareix 
        query_matrix = self.W_query(inputs)
        key_matrix = self.W_key(inputs)
        value_matrix = self.W_value(inputs)

        # scaling factor 
        d_k = key_matrix.shape[-1]


        # Calculate attention weights
        attension_scores = query_matrix @ key_matrix.transpose(1,2)

        # Applying causal mask -- IMPORTANT
        attension_scores.masked_fill_(self.mask.bool()[:num_token, :num_token], -torch.inf)
        #print(f"Masked attention score: {attension_scores}")

        attention_weights = torch.softmax(attension_scores/d_k**0.5, dim=-1)
        #print(f"Masked attention weights: {attention_weights}")

        # DROPOUT MASK
        attention_weights = self.dropout_mask(attention_weights)
        #print(f"Masked attention weights with dropout: {attention_weights}")

        # Calculate Context matrix
        context_matrix = attention_weights @ value_matrix
        return context_matrix

#print(batch.shape)
context_length = batch.shape[1]
self_attention = MyCausalSelfAttention_V2(d_in, d_out, context_length, dropout=0.5, bias=False)
self_attention(batch)

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


tensor([[[-0.9038,  0.4432],
         [-0.4368,  0.2142],
         [-0.4849, -0.1341],
         [-0.5834,  0.0081],
         [-0.6219, -0.0526],
         [-0.1417, -0.0505]],

        [[ 0.0000,  0.0000],
         [-1.1749,  0.0116],
         [-0.7733,  0.0073],
         [-0.9140, -0.2769],
         [-0.7679, -0.0735],
         [-0.6749, -0.0984]]], grad_fn=<UnsafeViewBackward0>)

In [15]:
mask = torch.triu(torch.ones(attension_scores.shape), diagonal=1)
masked_attension_scores = attension_scores.masked_fill(mask.bool(), -torch.inf)
mask.bool()

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

In [16]:
# Stacking multiple single head attention layer - MULTI HEAD ATTENTION
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout = 0.5, bias=False, num_heads=2):
        super().__init__()
        self.heads = nn.ModuleList([
            MyCausalSelfAttention_V2(d_in, d_out, context_length, dropout=0.5, bias=False) for _ in range(num_heads)
        ])

    def forward(self, x):
       return torch.cat([head(x) for head in self.heads], dim= -1)

torch.manual_seed(123)
context_length = batch.shape[1]
d_in = batch.shape[-1]
d_out = 2
multi_head_attention = MultiHeadAttention(d_in, d_out, context_length, dropout=0.5, bias=False, num_heads=2)
multi_head_attention(batch)

tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000],
         [-0.7381, -0.2026,  0.0000,  0.0000],
         [-0.9717, -0.2678,  0.9703,  0.7117],
         [-0.2210,  0.1084,  0.3492,  0.2569],
         [-0.4833, -0.1435,  0.4143,  0.3235],
         [-0.9130, -0.2881,  0.8947,  0.5938]],

        [[ 0.0000,  0.0000,  0.9544,  0.2127],
         [ 0.0000,  0.0000,  1.1781,  0.6513],
         [-0.7751,  0.0077,  0.4856,  0.3552],
         [-0.5504, -0.1770,  0.8795,  0.6697],
         [-0.8117, -0.1150,  0.4682,  0.3227],
         [-0.4257, -0.1539,  0.6311,  0.4446]]], grad_fn=<CatBackward0>)

In [17]:
# OPTIMIZING MULTI HEAD ATTENTION

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
